# SmartEvision ML Training Pipeline

## Naive Bayes Classifiers for Educational Document Analysis

**Capstone Project — Document Classification System**

This notebook trains three Multinomial Naive Bayes classifiers for automated
classification of DepEd educational documents:

1. **Subject Classifier** — Predicts learning area (English, Filipino, Mathematics, etc.)
2. **Grade Level Classifier** — Predicts grade level (Kindergarten to Grade 6)
3. **Document Type Classifier** — Distinguishes DLL, ISP, and ISR documents

All models use the ILAW-format training datasets and export JSON for browser-based inference.

---
## 1. Environment Setup

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
import re
import os
import warnings
from datetime import datetime
from collections import Counter

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_recall_fscore_support, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.size': 10
})
np.random.seed(42)

print('Environment ready.')
print(f'  NumPy: {np.__version__}')
print(f'  Pandas: {pd.__version__}')
print(f'  sklearn: imported')

ModuleNotFoundError: No module named 'numpy'

---
## 2. Configuration & Paths

In [ ]:
# Base directory (relative to notebook location)
BASE_DIR = os.path.abspath('')
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, '..', 'src', 'lib', 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Data directory:    {os.path.abspath(DATA_DIR)}')
print(f'Output directory:  {os.path.abspath(OUTPUT_DIR)}')
print(f'Results directory: {os.path.abspath(RESULTS_DIR)}')

---
## 3. Data Loading

Three CSV files serve as training corpora. Each row contains a text sample
and its corresponding label.

- `subject_training.csv`: text + subject label
- `gradelevel_training.csv`: text + grade level label
- `doctype_training.csv`: text + document type label

In [ ]:
def load_dataset(filename, label_col):
    """Load a CSV training dataset."""
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'{filepath} not found')
    df = pd.read_csv(filepath, header=0, names=['text', 'label'])
    df['label'] = df['label'].astype(str).str.strip()
    df['text'] = df['text'].astype(str).str.strip()
    df = df[df['text'].str.len() >= 10].reset_index(drop=True)
    print(f'  Loaded {len(df)} samples from {filename}')
    return df

datasets = {
    'subject': load_dataset('subject_training.csv', 'label'),
    'gradelevel': load_dataset('gradelevel_training.csv', 'label'),
    'doctype': load_dataset('doctype_training.csv', 'label'),
}

print(f'\nTotal training samples: {sum(len(d) for d in datasets.values())}')

---
## 4. Exploratory Data Analysis

### 4.1 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ['Subject Class Distribution', 'Grade Level Distribution', 'Document Type Distribution']
palettes = ['viridis', 'magma', 'rocket']

for ax, (key, df), title, pal in zip(axes, datasets.items(), titles, palettes):
    counts = df['label'].value_counts()
    colors = sns.color_palette(pal, n_colors=len(counts))
    bars = ax.barh(range(len(counts)), counts.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(counts)))
    ax.set_yticklabels(counts.index, fontsize=9)
    ax.set_xlabel('Sample Count')
    ax.set_title(title, fontsize=12, fontweight='bold')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=8)
    ax.set_xlim(0, counts.max() * 1.2)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_distributions.png'), dpi=150)
plt.show()
print('Class distribution charts saved to results/')

### 4.2 Text Length Analysis

Understanding the distribution of text lengths helps assess whether the
corpus has sufficient variation for the classifiers to learn from.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (key, df), title in zip(axes, datasets.items(),
    ['Subject — Text Length Distribution', 'Grade Level — Text Length Distribution',
     'Doc Type — Text Length Distribution']):
    lengths = df['text'].str.len()
    ax.hist(lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(lengths.median(), color='red', linestyle='--', label=f'Median: {lengths.median():.0f}')
    ax.set_xlabel('Text Length (characters)')
    ax.set_ylabel('Frequency')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'text_length_distributions.png'), dpi=150)
plt.show()

---
## 5. Tokenizer

The tokenizer must match the one used in the browser-side
`src/lib/utils/nlpClassifier.ts` to ensure consistent inference.

Strategy: lowercase, strip non-alphanumeric characters, split on whitespace.

In [ ]:
def tokenize(text):
    if not isinstance(text, str):
        return []
    return re.sub(r'[^a-z0-9\\s]', '', text.lower()).split()

# Quick test
print('Tokenize test:')
print(tokenize('Grade 2 DLL — Filipino (ILAW Format) WEEK 1'))

---
## 6. Training & Evaluation Function

Each classifier undergoes:
- **Training**: Multinomial Naive Bayes with CountVectorizer (unigrams)
- **Cross-validation**: 5-fold StratifiedKFold
- **Evaluation**: Confusion matrix, precision/recall/F1 per class, accuracy
- **Visualization**: Heatmap + per-class bar charts

In [ ]:
def train_evaluate(df, name):
    """Train a Naive Bayes classifier and produce evaluation outputs."""
    X = df['text']
    y = df['label']
    classes = sorted(y.unique())
    n_classes = len(classes)

    print(f'\\n{"="*60}')
    print(f'  {name}')
    print(f'  Samples: {len(df)}, Classes: {n_classes}')
    print(f'{"="*60}')

    # --- Vectorizer ---
    vectorizer = CountVectorizer(tokenizer=tokenize, token_pattern=None)
    X_counts = vectorizer.fit_transform(X)
    vocab_size = len(vectorizer.get_feature_names_out())
    print(f'  Vocabulary size: {vocab_size}')

    # --- Model ---
    clf = MultinomialNB(alpha=1.0)
    clf.fit(X_counts, y)

    y_pred = clf.predict(X_counts)
    train_acc = accuracy_score(y, y_pred)

    # --- Classification Report ---
    report_dict = classification_report(y, y_pred, target_names=classes, output_dict=True)
    print(f'\\n  Training Accuracy: {train_acc*100:.2f}%')
    print(f'\\n  Classification Report:')
    print(classification_report(y, y_pred, target_names=classes))

    # --- Cross-Validation ---
    k = min(5, len(df))
    if k < 2:
        k = 2
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    cv_scores = []
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        vec = CountVectorizer(tokenizer=tokenize, token_pattern=None)
        X_tr = vec.fit_transform(X_train)
        X_te = vec.transform(X_test)
        m = MultinomialNB(alpha=1.0)
        m.fit(X_tr, y_train)
        cv_scores.append(accuracy_score(y_test, m.predict(X_te)))
        print(f'  Fold {fold}: {cv_scores[-1]*100:.2f}%')

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))
    print(f'  Cross-Val: {cv_mean*100:.2f}% ± {cv_std*100:.2f}%')

    # --- Confusion Matrix ---
    cm = confusion_matrix(y, y_pred, labels=classes)
    fig, ax = plt.subplots(1, 1, figsize=(max(7, n_classes * 0.8), max(6, n_classes * 0.7)))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes,
                cmap='Blues', ax=ax, linewidths=0.5, linecolor='white')
    ax.set_title(f'{name} — Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    plt.tight_layout()
    cm_path = os.path.join(RESULTS_DIR, f'{name.lower().replace(" ", "_")}_cm.png')
    plt.savefig(cm_path, dpi=150)
    plt.show()

    # --- Per-Class Metrics Bar Chart ---
    prec, rec, f1, _ = precision_recall_fscore_support(y, y_pred, labels=classes)
    metrics_df = pd.DataFrame({'Precision': prec, 'Recall': rec, 'F1-Score': f1}, index=classes)
    fig, ax = plt.subplots(figsize=(max(10, n_classes * 0.6), 5))
    metrics_df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='white', width=0.8)
    ax.set_title(f'{name} — Per-Class Metrics', fontsize=13, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.legend(loc='lower right', fontsize=9)
    ax.set_xlabel('')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.tight_layout()
    bar_path = os.path.join(RESULTS_DIR, f'{name.lower().replace(" ", "_")}_metrics.png')
    plt.savefig(bar_path, dpi=150)
    plt.show()

    # --- Top Predictive Words Per Class ---
    feature_names = vectorizer.get_feature_names_out()
    top_words = {}
    for i, label in enumerate(classes):
        log_probs = clf.feature_log_prob_[i]
        top_indices = np.argsort(log_probs)[-15:][::-1]
        top_words[label] = [feature_names[j] for j in top_indices]

    # --- Top words table ---
    fig, ax = plt.subplots(figsize=(max(10, n_classes * 2), max(4, n_classes * 0.4)))
    ax.axis('off')
    table_data = []
    for label, words in top_words.items():
        table_data.append([label, ', '.join(words)])
    table = ax.table(cellText=table_data, colLabels=['Class', 'Top 15 Predictive Words'],
                     cellLoc='left', loc='center', colWidths=[0.18, 0.72])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.4)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor('navy')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#f0f0f0')
    ax.set_title(f'{name} — Top Predictive Words per Class', fontsize=13, fontweight='bold', pad=20)
    plt.tight_layout()
    tw_path = os.path.join(RESULTS_DIR, f'{name.lower().replace(" ", "_")}_top_words.png')
    plt.savefig(tw_path, dpi=150)
    plt.show()

    # --- Save per-class metrics CSV ---
    metrics_df.to_csv(os.path.join(RESULTS_DIR, f'{name.lower().replace(" ", "_")}_metrics.csv'))

    return {
        'classifier': clf,
        'vectorizer': vectorizer,
        'labels': classes,
        'accuracy': float(train_acc),
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'report': report_dict,
        'confusion_matrix': cm.tolist(),
        'n_samples': len(df),
        'vocabulary_size': vocab_size,
        'top_words': top_words,
        'metrics_df': metrics_df,
    }

print('Training function defined.')

---
## 7. Subject Classifier

In [ ]:
subject_result = train_evaluate(datasets['subject'], 'Subject Classifier')

---
## 8. Grade Level Classifier

In [ ]:
grade_result = train_evaluate(datasets['gradelevel'], 'Grade Level Classifier')

---
## 9. Document Type Classifier

In [ ]:
doctype_result = train_evaluate(datasets['doctype'], 'Document Type Classifier')

---
## 10. Comparative Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Training Accuracy', 'Cross-Val Mean', 'Cross-Val Std',
               'Num Samples', 'Num Classes', 'Vocabulary Size'],
    'Subject Classifier': [
        f'{subject_result["accuracy"]*100:.2f}%',
        f'{subject_result["cv_mean"]*100:.2f}%',
        f'{subject_result["cv_std"]*100:.2f}%',
        str(subject_result['n_samples']),
        str(len(subject_result['labels'])),
        str(subject_result['vocabulary_size']),
    ],
    'Grade Level Classifier': [
        f'{grade_result["accuracy"]*100:.2f}%',
        f'{grade_result["cv_mean"]*100:.2f}%',
        f'{grade_result["cv_std"]*100:.2f}%',
        str(grade_result['n_samples']),
        str(len(grade_result['labels'])),
        str(grade_result['vocabulary_size']),
    ],
    'Doc Type Classifier': [
        f'{doctype_result["accuracy"]*100:.2f}%',
        f'{doctype_result["cv_mean"]*100:.2f}%',
        f'{doctype_result["cv_std"]*100:.2f}%',
        str(doctype_result['n_samples']),
        str(len(doctype_result['labels'])),
        str(doctype_result['vocabulary_size']),
    ],
}).set_index('Metric')

print('=== PERFORMANCE SUMMARY ===')
display(summary)

# Save summary
summary.to_csv(os.path.join(RESULTS_DIR, 'performance_summary.csv'))
print('Summary saved to results/performance_summary.csv')

### 10.1 Comparative Accuracy Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
names = ['Subject', 'Grade Level', 'Doc Type']
accs = [subject_result['accuracy'], grade_result['accuracy'], doctype_result['accuracy']]
cv_means = [subject_result['cv_mean'], grade_result['cv_mean'], doctype_result['cv_mean']]
cv_stds = [subject_result['cv_std'], grade_result['cv_std'], doctype_result['cv_std']]

x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, [a*100 for a in accs], width, label='Training Accuracy',
               color='steelblue', edgecolor='white')
bars2 = ax.bar(x + width/2, [c*100 for c in cv_means], width, label='Cross-Val Mean',
               color='coral', edgecolor='white', yerr=[s*100 for s in cv_stds], capsize=4)

ax.set_ylabel('Accuracy (%)')
ax.set_title('Classifier Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 105)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'comparative_accuracy.png'), dpi=150)
plt.show()

---
## 11. Export Models to JSON

The trained models are exported as JSON files compatible with
`src/lib/utils/nlpClassifier.ts` for browser-side inference.

In [ ]:
def export_to_json(result, filename):
    """Export trained classifier to JSON for TypeScript inference."""
    clf = result['classifier']
    vectorizer = result['vectorizer']
    vocab = vectorizer.get_feature_names_out()
    classes = clf.classes_

    model = {'classes': {}, 'vocabularySize': len(vocab)}
    for i, label in enumerate(classes):
        word_probs = {}
        for j, word in enumerate(vocab):
            word_probs[word] = float(clf.feature_log_prob_[i][j])
        total_words = float(np.sum(clf.feature_count_[i]))
        default_prob = float(np.log(1.0 / (total_words + len(vocab))))
        model['classes'][label] = {
            'priorProbability': float(clf.class_log_prior_[i]),
            'wordProbabilities': word_probs,
            'defaultWordProb': default_prob,
        }

    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(model, f, indent=2)
    size_kb = os.path.getsize(filepath) / 1024
    print(f'  Exported: {filepath} ({size_kb:.1f} KB)')
    return filepath

print('Exporting models...')
subject_model = export_to_json(subject_result, 'subject_classifier_model.json')
grade_model = export_to_json(grade_result, 'gradelevel_classifier_model.json')
doctype_model = export_to_json(doctype_result, 'doctype_classifier_model.json')
print('\\nAll models exported successfully.')

---
## 12. Save Complete Training Results

All metrics, confusion matrices, and metadata are saved to
`results/training_results.json` for capstone documentation.

In [ ]:
def convert_report(report_dict):
    """Convert numpy values in report to native Python types."""
    cleaned = {}
    for k, v in report_dict.items():
        if isinstance(v, dict):
            cleaned[k] = {kk: float(vv) if isinstance(vv, (np.floating, float)) else vv
                          for kk, vv in v.items()}
        else:
            cleaned[k] = float(v) if isinstance(v, (np.floating, float)) else v
    return cleaned

training_results = {
    'training_date': datetime.now().isoformat(),
    'total_samples': sum(len(d) for d in datasets.values()),
    'datasets': {k: {'samples': len(v), 'classes': sorted(v['label'].unique()).tolist()}
                 for k, v in datasets.items()},
    'subject': {
        'n_samples': subject_result['n_samples'],
        'vocabulary_size': subject_result['vocabulary_size'],
        'accuracy': round(subject_result['accuracy'] * 100, 2),
        'cv_mean': round(subject_result['cv_mean'] * 100, 2),
        'cv_std': round(subject_result['cv_std'] * 100, 2),
        'report': convert_report(subject_result['report']),
        'confusion_matrix': subject_result['confusion_matrix'],
        'top_words': {k: v for k, v in subject_result['top_words'].items()},
    },
    'grade_level': {
        'n_samples': grade_result['n_samples'],
        'vocabulary_size': grade_result['vocabulary_size'],
        'accuracy': round(grade_result['accuracy'] * 100, 2),
        'cv_mean': round(grade_result['cv_mean'] * 100, 2),
        'cv_std': round(grade_result['cv_std'] * 100, 2),
        'report': convert_report(grade_result['report']),
        'confusion_matrix': grade_result['confusion_matrix'],
        'top_words': {k: v for k, v in grade_result['top_words'].items()},
    },
    'doc_type': {
        'n_samples': doctype_result['n_samples'],
        'vocabulary_size': doctype_result['vocabulary_size'],
        'accuracy': round(doctype_result['accuracy'] * 100, 2),
        'cv_mean': round(doctype_result['cv_mean'] * 100, 2),
        'cv_std': round(doctype_result['cv_std'] * 100, 2),
        'report': convert_report(doctype_result['report']),
        'confusion_matrix': doctype_result['confusion_matrix'],
        'top_words': {k: v for k, v in doctype_result['top_words'].items()},
    },
}

results_path = os.path.join(RESULTS_DIR, 'training_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(training_results, f, indent=2)
print(f'Training results saved to {results_path}')
print(f'\\n{"="*60}')
print(f'  TRAINING COMPLETE')
print(f'{"="*60}')
print(f'  Subject     : {training_results["subject"]["accuracy"]}% '
      f'(CV: {training_results["subject"]["cv_mean"]}%)')
print(f'  Grade Level : {training_results["grade_level"]["accuracy"]}% '
      f'(CV: {training_results["grade_level"]["cv_mean"]}%)')
print(f'  Doc Type    : {training_results["doc_type"]["accuracy"]}% '
      f'(CV: {training_results["doc_type"]["cv_mean"]}%)')
print(f'  Models saved to: {OUTPUT_DIR}')
print(f'  Results saved to: {RESULTS_DIR}')

---
## 13. Conclusion

### Summary of Generated Files

| File | Description |
|------|-------------|
| `src/lib/models/subject_classifier_model.json` | Subject classifier for browser inference |
| `src/lib/models/gradelevel_classifier_model.json` | Grade level classifier for browser inference |
| `src/lib/models/doctype_classifier_model.json` | Document type classifier for browser inference |
| `results/training_results.json` | Complete metrics for documentation |
| `results/performance_summary.csv` | Comparative summary table |
| `results/*_cm.png` | Confusion matrix heatmaps |
| `results/*_metrics.png` | Per-class precision/recall/F1 bar charts |
| `results/*_metrics.csv` | Per-class metrics in CSV format |
| `results/*_top_words.png` | Top predictive words per class |
| `results/class_distributions.png` | Class distribution overview |
| `results/text_length_distributions.png` | Text length analysis |
| `results/comparative_accuracy.png` | Cross-classifier accuracy comparison |

### Key Takeaways

- All three classifiers use Multinomial Naive Bayes with Laplace smoothing
- The tokenizer matches the browser-side implementation for consistent inference
- Cross-validation confirms model generalizability beyond training data
- Per-class metrics identify which categories are easiest vs hardest to classify
- Top predictive words provide interpretability of model decisions